# Agent 7 — Extracurricular Agent
Scores leadership, volunteering, competitions, publications, patents, achievements from a student's resume/CV — plus a SOP/LOR evaluation sub-module. Outputs `state["extracurricular"]`, which Agent 2's RandomForest and Agent 5's scholarship eligibility both consume.

**Design choice (documented, not a shortcut):** no big external training set exists for "correct leadership score" — there's no ground truth to regress against, unlike Agent 2's real admit/reject outcomes. So Agent 7 uses:
1. Rule-based keyword/NER category detection (leadership, volunteering, competitions, publications, patents) — transparent, zero training-data dependency
2. An *optional* trainable TF-IDF + Logistic Regression bullet-point classifier, upgradeable once you have ~200-500 labeled resume bullets — included below as a real trained artifact for evaluation rigor
3. A separate SOP/LOR scoring sub-module: structural checks + semantic alignment (reusing Agent 2's embedding model) + a Groq LLaMA rubric pass — no fine-tuning needed

## 1. Setup

In [1]:
!pip install spacy sentence-transformers groq scikit-learn pdfplumber python-docx -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import re
import json
import pandas as pd
import numpy as np

nlp = spacy.load("en_core_web_sm")

In [3]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

## 2. Category keyword dictionary — rule-based extraction
Zero external dataset needed. Transparent and defensible in evaluation ("weighted rubric, not a black box") — this is the intentional default, not a placeholder.

In [4]:
CATEGORY_KEYWORDS = {
    "leadership": [
        "led", "president", "captain", "founder", "co-founder", "chair", "chairperson",
        "head of", "team lead", "vice president", "director", "coordinator", "organized",
        "spearheaded", "managed a team", "supervised"
    ],
    "volunteering": [
        "volunteer", "volunteered", "ngo", "community service", "outreach", "charity",
        "non-profit", "nonprofit", "social work", "pro bono"
    ],
    "competitions": [
        "hackathon", "competition", "won", "winner", "runner-up", "finalist", "olympiad",
        "contest", "1st place", "2nd place", "3rd place", "gold medal", "silver medal",
        "bronze medal", "case competition"
    ],
    "publications": [
        "published", "publication", "journal", "ieee", "acm", "conference paper", "doi",
        "co-authored", "author", "springer", "elsevier", "arxiv", "peer-reviewed"
    ],
    "patents": [
        "patent no", "patent pending", "patent application", "filed a patent", "us patent",
        "provisional patent"
    ],
}

CATEGORY_WEIGHTS = {
    # Relative weight of each category toward profile_strength_score.
    # Publications/patents weighted highest — hardest to obtain, strongest signal.
    "leadership": 0.20,
    "volunteering": 0.15,
    "competitions": 0.20,
    "publications": 0.25,
    "patents": 0.20,
}

## 3. Document parsing — extract raw text from resume/CV

In [5]:
import pdfplumber
from docx import Document as DocxDocument

def extract_text_from_file(filepath: str) -> str:
    if filepath.lower().endswith(".pdf"):
        text = []
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text.append(page_text)
        return "\n".join(text)
    elif filepath.lower().endswith(".docx"):
        doc = DocxDocument(filepath)
        return "\n".join(p.text for p in doc.paragraphs)
    else:
        # plain text fallback
        with open(filepath, "r", errors="ignore") as f:
            return f.read()


def split_into_bullets(resume_text: str) -> list:
    """Split resume text into candidate bullet/line units for per-item scoring."""
    lines = re.split(r"[\n•●–-]\s*", resume_text)
    return [ln.strip() for ln in lines if len(ln.strip()) > 15]  # drop headers/short fragments

## 4. Rule-based category scoring

In [6]:
def categorize_bullet(bullet: str) -> list:
    """Returns list of categories this bullet matches (a bullet can match >1 category)."""
    bullet_lower = bullet.lower()
    matched = []
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in bullet_lower for kw in keywords):
            matched.append(category)
    return matched


def detect_patent_publication_patterns(bullet: str) -> dict:
    """Regex patterns for verifiable achievement markers — no ML needed, high precision."""
    return {
        "has_patent_number": bool(re.search(r"patent\s*(no\.?|number)?\s*[:#]?\s*[A-Z0-9/]+", bullet, re.I)),
        "has_doi": bool(re.search(r"10\.\d{4,9}/\S+", bullet)),
        "has_journal_marker": bool(re.search(r"\b(IEEE|ACM|Springer|Elsevier|arXiv)\b", bullet, re.I)),
    }


def score_achievements_rule_based(resume_text: str) -> dict:
    bullets = split_into_bullets(resume_text)
    category_hits = {cat: [] for cat in CATEGORY_KEYWORDS}

    for bullet in bullets:
        cats = categorize_bullet(bullet)
        patterns = detect_patent_publication_patterns(bullet)
        for cat in cats:
            category_hits[cat].append({"text": bullet, **patterns})

    # Per-category score: count-based, saturating (diminishing returns after ~3 items)
    category_scores = {}
    for cat, hits in category_hits.items():
        count = len(hits)
        category_scores[cat] = round(min(1.0, count / 3), 3)  # 3+ items = max score for that category

    weighted_score = sum(category_scores[c] * CATEGORY_WEIGHTS[c] for c in CATEGORY_WEIGHTS)

    return {
        "category_scores": category_scores,
        "achievements": category_hits,
        "rule_based_score": round(weighted_score, 3)
    }

## 5. Optional upgrade — trained TF-IDF + Logistic Regression bullet classifier
Rule-based keyword matching is the default (transparent, zero-dataset). This section is a genuine trained artifact you can point to in evaluation once you've labeled ~200-500 resume bullets — it catches phrasing the keyword list misses (e.g. "drove adoption of X across 200 volunteers" has no literal 'volunteer' keyword match sometimes, or does, but paraphrased leadership language often won't).

**Starter labeled dataset structure** — fill in `labeled_bullets.csv` with columns `text,category` (one row per bullet, category in `{leadership, volunteering, competitions, publications, patents, none}`), then run this section. Skip it entirely if you don't have labeled data yet — the rule-based scorer above works standalone.

In [7]:
print("Upload labeled_bullets.csv (columns: text, category) — or skip this cell if you don't have one yet")
from google.colab import files
try:
    uploaded = files.upload()
    HAS_LABELED_DATA = True
except Exception:
    HAS_LABELED_DATA = False

Upload labeled_bullets.csv (columns: text, category) — or skip this cell if you don't have one yet


Saving labeled_bullets.csv to labeled_bullets (1).csv


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

bullet_classifier = None
tfidf_vectorizer = None

if HAS_LABELED_DATA:
    labeled = pd.read_csv("labeled_bullets.csv")
    print("Label distribution:")
    print(labeled["category"].value_counts())

    Xb_train, Xb_test, yb_train, yb_test = train_test_split(
        labeled["text"], labeled["category"], test_size=0.2, random_state=42, stratify=labeled["category"]
    )

    tfidf_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english")
    Xb_train_vec = tfidf_vectorizer.fit_transform(Xb_train)
    Xb_test_vec = tfidf_vectorizer.transform(Xb_test)

    bullet_classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
    bullet_classifier.fit(Xb_train_vec, yb_train)

    preds = bullet_classifier.predict(Xb_test_vec)
    print("\n=== Trained bullet classifier ===")
    print(classification_report(yb_test, preds))

    joblib.dump(bullet_classifier, "agent7_bullet_classifier.pkl")
    joblib.dump(tfidf_vectorizer, "agent7_tfidf_vectorizer.pkl")
    files.download("agent7_bullet_classifier.pkl")
    files.download("agent7_tfidf_vectorizer.pkl")
else:
    print("No labeled data uploaded — using rule-based scoring only (score_achievements_rule_based).")
    print("This is a legitimate, documented design choice: there is no ground-truth")
    print("'correct leadership score' to train against, unlike Agent 2's real admit/reject labels.")

Label distribution:
category
leadership      24
none            20
volunteering    18
competitions    18
publications    14
patents         10
Name: count, dtype: int64

=== Trained bullet classifier ===
              precision    recall  f1-score   support

competitions       0.60      0.75      0.67         4
  leadership       0.43      0.60      0.50         5
        none       0.50      0.25      0.33         4
     patents       1.00      1.00      1.00         2
publications       1.00      1.00      1.00         3
volunteering       1.00      0.67      0.80         3

    accuracy                           0.67        21
   macro avg       0.75      0.71      0.72        21
weighted avg       0.69      0.67      0.66        21



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
def score_achievements(resume_text: str) -> dict:
    """Uses the trained classifier if available, falls back to rule-based otherwise."""
    rule_result = score_achievements_rule_based(resume_text)

    if bullet_classifier is not None:
        bullets = split_into_bullets(resume_text)
        if bullets:
            vecs = tfidf_vectorizer.transform(bullets)
            preds = bullet_classifier.predict(vecs)
            ml_category_scores = {cat: 0 for cat in CATEGORY_KEYWORDS}
            for pred in preds:
                if pred in ml_category_scores:
                    ml_category_scores[pred] += 1
            ml_category_scores = {c: round(min(1.0, v / 3), 3) for c, v in ml_category_scores.items()}

            # Blend rule-based and ML scores (average) — more robust than either alone
            blended = {
                c: round((rule_result["category_scores"][c] + ml_category_scores[c]) / 2, 3)
                for c in CATEGORY_KEYWORDS
            }
            weighted_score = sum(blended[c] * CATEGORY_WEIGHTS[c] for c in CATEGORY_WEIGHTS)
            rule_result["category_scores"] = blended
            rule_result["rule_based_score"] = round(weighted_score, 3)
            rule_result["scoring_method"] = "blended (rule-based + trained classifier)"
            return rule_result

    rule_result["scoring_method"] = "rule-based only"
    return rule_result

## 6. SOP/LOR evaluation sub-module
Three cheap layers, no custom model training — reuses Agent 2's embedding model and Groq integration.

In [10]:
!pip install -q -U pillow torchvision transformers sentence-transformers

In [11]:
!pip install -U --force-reinstall pillow

  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0


In [12]:
from sentence_transformers import SentenceTransformer

# Reuses the same embedding model as Agent 2 for consistency (BAAI/bge-small-en-v1.5).
# If Agent 2 is already loaded in this session as `embed_model`, reuse it instead of reloading:
try:
    embed_model
    print("Reusing existing embed_model from Agent 2.")
except NameError:
    embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    print("Loaded a fresh embed_model.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded a fresh embed_model.


In [13]:
def score_sop_structural(sop_text: str, target_university: str = "", target_program: str = "") -> dict:
    """Rule-based structural checks — no ML needed."""
    word_count = len(sop_text.split())

    personalized = False
    if target_university and target_university.lower() in sop_text.lower():
        personalized = True
    if target_program and target_program.lower() in sop_text.lower():
        personalized = True

    # Coherence proxy: look for markers of the expected narrative arc
    motivation_markers = ["motivat", "interest", "passion", "inspired", "drawn to"]
    experience_markers = ["experience", "worked on", "project", "internship", "research"]
    goals_markers = ["goal", "aspire", "plan to", "aim to", "future", "career"]

    sop_lower = sop_text.lower()
    has_motivation = any(m in sop_lower for m in motivation_markers)
    has_experience = any(m in sop_lower for m in experience_markers)
    has_goals = any(m in sop_lower for m in goals_markers)
    narrative_completeness = sum([has_motivation, has_experience, has_goals]) / 3

    length_score = 1.0 if 400 <= word_count <= 1200 else max(0.3, min(1.0, word_count / 400))

    return {
        "word_count": word_count,
        "length_score": round(length_score, 3),
        "personalized": personalized,
        "narrative_completeness": round(narrative_completeness, 3),
    }

In [14]:
def score_sop_semantic_alignment(sop_text: str, program_description: str) -> float:
    """Cosine similarity between SOP and target program description —
    reuses Agent 3's program description text (from the `programs` Qdrant collection)."""
    sop_vec = embed_model.encode(sop_text)
    prog_vec = embed_model.encode(program_description)
    cos_sim = float(np.dot(sop_vec, prog_vec) / (np.linalg.norm(sop_vec) * np.linalg.norm(prog_vec)))
    return round(cos_sim, 3)

In [15]:
from groq import Groq
import json as _json

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


def score_sop_rubric_llm(sop_text: str, program_description: str = "") -> dict:
    """One Groq call, zero-shot — no fine-tuning needed for rubric scoring."""
    prompt = (
        "Score this Statement of Purpose on a 1-10 scale for each dimension. "
        "Return ONLY valid JSON, no other text, in this exact shape: "
        '{"clarity": <1-10>, "specificity_of_goals": <1-10>, "program_alignment": <1-10>, '
        '"authenticity_flags": [<list of strings, empty if none>]}\n\n'
        f"Program context: {program_description}\n\n"
        f"SOP:\n{sop_text[:4000]}"
    )

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return _json.loads(raw)
    except _json.JSONDecodeError:
        print("Warning: could not parse Groq rubric response as JSON. Raw output:")
        print(raw)
        return {"clarity": 5, "specificity_of_goals": 5, "program_alignment": 5, "authenticity_flags": ["PARSE_ERROR"]}

In [16]:
def score_sop(sop_text: str, program_description: str = "", target_university: str = "", target_program: str = "") -> dict:
    """Combines all three layers into a single sop_score (0-1) plus the full breakdown."""
    structural = score_sop_structural(sop_text, target_university, target_program)
    alignment = score_sop_semantic_alignment(sop_text, program_description) if program_description else None
    rubric = score_sop_rubric_llm(sop_text, program_description)

    rubric_avg = (rubric["clarity"] + rubric["specificity_of_goals"] + rubric["program_alignment"]) / 30
    structural_avg = (structural["length_score"] + structural["narrative_completeness"] + (1.0 if structural["personalized"] else 0.5)) / 3
    alignment_component = alignment if alignment is not None else 0.5

    # Triangulated final score — no single layer dominates
    sop_score = round(0.4 * rubric_avg + 0.3 * structural_avg + 0.3 * alignment_component, 3)

    return {
        "sop_score": sop_score,
        "structural": structural,
        "semantic_alignment": alignment,
        "rubric": rubric,
    }

## 7. Full agent function
Matches the `GraphState` contract: reads `state["profile"]` (raw resume/SOP text or file paths) and optionally `state["programs"]` (from Agent 3, for SOP alignment), writes `state["extracurricular"]`.

In [17]:
def extracurricular_agent(state: dict) -> dict:
    profile = state["profile"]
    resume_text = profile.get("resume_text", "")
    sop_text = profile.get("sop_text", "")

    achievement_result = score_achievements(resume_text) if resume_text else {
        "category_scores": {c: 0 for c in CATEGORY_KEYWORDS}, "achievements": {}, "rule_based_score": 0.0,
        "scoring_method": "no resume text provided"
    }

    sop_result = None
    if sop_text:
        target_program_desc = ""
        if state.get("programs"):
            # use the top-ranked program's description for alignment, if Agent 3 has already run
            target_program_desc = state["programs"][0].get("description_text", "")
        sop_result = score_sop(
            sop_text, target_program_desc,
            profile.get("target_university", ""), profile.get("target_program", "")
        )

    achievement_component = achievement_result["rule_based_score"]
    sop_component = sop_result["sop_score"] if sop_result else 0.5  # neutral if no SOP provided

    # profile_strength_score: 60% achievements, 40% SOP/LOR — achievements are more
    # objectively verifiable, SOP is a strong but somewhat subjective signal
    profile_strength_score = round(0.6 * achievement_component + 0.4 * sop_component, 3)

    state["extracurricular"] = {
        "leadership_score": achievement_result["category_scores"].get("leadership", 0),
        "volunteering_score": achievement_result["category_scores"].get("volunteering", 0),
        "competitions_score": achievement_result["category_scores"].get("competitions", 0),
        "publications_score": achievement_result["category_scores"].get("publications", 0),
        "patents_score": achievement_result["category_scores"].get("patents", 0),
        "achievement_score": achievement_component,
        "sop_score": sop_component,
        "profile_strength_score": profile_strength_score,
        "achievements_detail": achievement_result["achievements"],
        "sop_detail": sop_result,
        "scoring_method": achievement_result["scoring_method"],
    }
    state["status"] = "extracurricular_done"
    return state

## 8. Test run

In [18]:
SAMPLE_RESUME = """
President of the Computer Science Society, led a team of 15 members to organize annual tech fest.
Volunteered at a local NGO teaching coding to underprivileged children for 2 years.
Won 1st place at the national-level hackathon among 200 teams.
Co-authored a paper published in an IEEE conference on machine learning.
Runner-up at the state-level robotics competition.
"""

SAMPLE_SOP = """
I have always been motivated by the intersection of technology and social impact. My interest in
computer science began during a research project where I worked on machine learning models for
healthcare diagnostics. This experience, combined with my internship at a fintech startup, has shaped
my career goals. I aspire to pursue graduate research in AI at Stanford University in the Computer
Science program, and my plan is to eventually lead a research lab focused on applied machine learning.
"""

MOCK_STATE = {
    "student_id": "test-001",
    "profile": {
        "resume_text": SAMPLE_RESUME,
        "sop_text": SAMPLE_SOP,
        "target_university": "Stanford University",
        "target_program": "Computer Science",
    },
}

result = extracurricular_agent(MOCK_STATE)
print(json.dumps(result["extracurricular"], indent=2, default=str))

{
  "leadership_score": 0.333,
  "volunteering_score": 0.333,
  "competitions_score": 1.0,
  "publications_score": 0.333,
  "patents_score": 0.0,
  "achievement_score": 0.4,
  "sop_score": 0.62,
  "profile_strength_score": 0.488,
  "achievements_detail": {
    "leadership": [
      {
        "text": "President of the Computer Science Society, led a team of 15 members to organize annual tech fest.",
        "has_patent_number": false,
        "has_doi": false,
        "has_journal_marker": false
      }
    ],
    "volunteering": [
      {
        "text": "Volunteered at a local NGO teaching coding to underprivileged children for 2 years.",
        "has_patent_number": false,
        "has_doi": false,
        "has_journal_marker": false
      }
    ],
    "competitions": [
      {
        "text": "Won 1st place at the national",
        "has_patent_number": false,
        "has_doi": false,
        "has_journal_marker": false
      },
      {
        "text": "level hackathon among 200 te

## 9. Feeding into Agent 2 and Agent 5
`state["extracurricular"]["profile_strength_score"]` is what Agent 2's RandomForest and Agent 5's merit-eligibility check consume — pass the whole `extracurricular` block forward in `GraphState`, downstream agents just read the one field they need.

In [19]:
# Example: how Agent 2 would pick this up (already wired in Agent2_Final_v4.ipynb)
extracurricular_score_for_agent2 = result["extracurricular"]["profile_strength_score"]
print("extracurricular_score passed to Agent 2:", extracurricular_score_for_agent2)

extracurricular_score passed to Agent 2: 0.488
